# Single-Category Regime Experiments

This notebook narrows the regime-based forecasting study to a single aligned category family across the three datasets.

Aligned family used for the paper:
- `M5 / Walmart`: `HOUSEHOLD`
- `Favorita`: `CLEANING`
- `Amazon`: `Health_and_Household`

This is not the exact same taxonomy label across datasets, but it is a defensible cross-market alignment around household and cleaning demand.


## Why a Single Category Family?

For the paper, using a single aligned category family makes the comparison cleaner:

- it reduces cross-category noise
- it makes the regime argument easier to defend
- it allows product examples to be narrated more clearly
- it makes `SARIMAX`, `HURDLE`, and `TSB` comparisons more interpretable


In [ ]:
import pandas as pd

category_alignment = pd.DataFrame([
    {
        'dataset': 'M5_WALMART',
        'category_used': 'HOUSEHOLD',
        'paper_family': 'Household / Cleaning / Hygiene',
    },
    {
        'dataset': 'FAVORITA',
        'category_used': 'CLEANING',
        'paper_family': 'Household / Cleaning / Hygiene',
    },
    {
        'dataset': 'AMAZON_2023',
        'category_used': 'Health_and_Household',
        'paper_family': 'Household / Cleaning / Hygiene',
    },
])

category_alignment


## Product Semantics

The raw IDs are not paper-friendly by themselves, so the study should attach a semantic product description whenever possible.

These labels should be treated as **proxy semantic labels** inferred from the cross-market alignment process and not as guaranteed SKU descriptions.


In [ ]:
semantic_product_map = pd.DataFrame([
    {
        'semantic_product': 'Paper roll (proxy)',
        'dataset': 'AMAZON_2023',
        'series_id': 'B0C1G1BJ2B',
        'product_type_note': 'paper / paper roll / tissue-like household paper product',
    },
    {
        'semantic_product': 'Soap / cleaner (proxy)',
        'dataset': 'AMAZON_2023',
        'series_id': 'B072KDRZH5',
        'product_type_note': 'soap / cleaner / home cleaning product',
    },
    {
        'semantic_product': 'Personal care (proxy)',
        'dataset': 'AMAZON_2023',
        'series_id': 'B09KC8ZYYX',
        'product_type_note': 'personal care / hygiene item',
    },
    {
        'semantic_product': 'Intermittent stress item (proxy)',
        'dataset': 'AMAZON_2023',
        'series_id': 'B0BZTLKX7H',
        'product_type_note': 'very-low-demand household/hygiene item under sparse demand',
    },
    {
        'semantic_product': 'Low-demand intermittent item (proxy)',
        'dataset': 'AMAZON_2023',
        'series_id': 'B00QWO9P0O',
        'product_type_note': 'low-rotation household/hygiene product',
    },
])

semantic_product_map


## Important Interpretation for TSB

A category scan showed that **Favorita / CLEANING is not the strongest dataset to showcase TSB**. In this aligned family, the strongest intermittent-demand examples come mainly from **Amazon Health_and_Household** and some **M5 HOUSEHOLD** SKUs.

Therefore:
- Favorita is kept mainly to illustrate `SARIMAX` and `HURDLE` cases
- Amazon and M5 are used to illustrate the strongest `TSB` cases


## Recommended Five Paper Cases Inside the Aligned Category Family

These five cases are selected to support a regime-driven story while staying inside the same broad category family.


In [ ]:
paper_cases_single_category = pd.DataFrame([
    {
        'case_id': 1,
        'dataset': 'FAVORITA',
        'series_id': 'item_268443',
        'semantic_product': 'Cleaning product (proxy high-demand example)',
        'expected_regime': 'stable / high-demand',
        'preferred_model': 'SARIMAX',
        'why': 'Dense and stable demand should favor a continuous time-series model with temporal structure.',
    },
    {
        'case_id': 2,
        'dataset': 'FAVORITA',
        'series_id': 'item_789224',
        'semantic_product': 'Cleaning product (proxy transition example)',
        'expected_regime': 'medium-demand / variable / transition',
        'preferred_model': 'HURDLE',
        'why': 'Transition behavior with possible irregular occurrence is a good setting for Hurdle.',
    },
    {
        'case_id': 3,
        'dataset': 'M5_WALMART',
        'series_id': 'HOUSEHOLD_1_187_WI_1_validation',
        'semantic_product': 'Household item (proxy transition retail example)',
        'expected_regime': 'medium-demand / transition',
        'preferred_model': 'HURDLE',
        'why': 'Already identified in the repo as a transition regime example.',
    },
    {
        'case_id': 4,
        'dataset': 'AMAZON_2023',
        'series_id': 'B0BR4W8TBH',
        'semantic_product': 'Personal care / hygiene item (very sparse TSB-favorable case)',
        'expected_regime': 'very-low-demand / intermittent extreme',
        'preferred_model': 'TSB',
        'why': 'Under the current notebook evaluation, this case clearly improves in both daily and weekly comparisons for TSB walk-forward.',
    },
    {
        'case_id': 5,
        'dataset': 'AMAZON_2023',
        'series_id': 'B0BZTL57BP',
        'semantic_product': 'Household / hygiene item (very sparse TSB-favorable case)',
        'expected_regime': 'very-low-demand / intermittent extreme',
        'preferred_model': 'TSB',
        'why': 'This Amazon series is strongly intermittent and gives TSB a clearer advantage than the weaker earlier showcase items.',
    },
    {
        'case_id': 6,
        'dataset': 'AMAZON_2023',
        'series_id': 'B0C1RRLY4Y',
        'semantic_product': 'Household / hygiene item (weekly TSB-favorable case)',
        'expected_regime': 'very-low-demand / intermittent extreme',
        'preferred_model': 'TSB',
        'why': 'It remains useful because TSB becomes more competitive after weekly aggregation even if daily MAE is close.',
    },
])

paper_cases_single_category


## Recommended Experimental Framing

### Framing sentence for the paper

A possible sentence for the paper is:

> We aligned a single household-oriented category family across datasets, using HOUSEHOLD in M5, CLEANING in Favorita, and Health_and_Household in Amazon, in order to isolate regime-dependent forecasting behavior within comparable product contexts.

### Product narration

When writing the paper, prefer semantic labels such as:
- paper roll
- soap / cleaner
- personal care item
- household item

instead of only raw IDs, and add `(proxy)` when exact category identity is inferred rather than explicitly validated.


## Model Expectations Within This Category Family

- `SARIMAX` is expected to work best on stable cleaning/household products with dense demand.
- `HURDLE` is expected to work better when cleaning/household demand becomes irregular or zero-heavy but still benefits from temporal covariates.
- `TSB` is expected to work best for very sparse household/hygiene items with intermittent purchase patterns.


## Figures to Generate for This Single-Category Study

For each selected case:

1. **Demand regime figure**
   - raw demand time series
   - zero-rate
   - ADI
   - CV²
   - regime label

2. **Forecast comparison figure**
   - actual demand
   - `SARIMAX`
   - `HURDLE`
   - `TSB`
   - note the preferred model for that regime

3. **Paper summary figure**
   - grouped bar chart of error by model and regime
   - final hybrid strategy diagram
